<a href="https://colab.research.google.com/github/MMujtabaX/agentic-ai-journey/blob/main/05_crewai_multi_agent/crewai_article_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Multi-Agent Article Generator with CrewAI + Gradio

---

## 📌 Overview

This notebook demonstrates a **production-style multi-agent workflow** using:

| Tool | Purpose |
|------|---------|
| 🧠 **CrewAI** | Orchestrates multiple AI agents with defined roles and tasks |
| 🔗 **LangChain + OpenAI** | Powers each agent with GPT-4o |
| 🌐 **SerperDev** | Gives the Researcher agent real-time web search |
| 🎨 **Gradio** | Provides an interactive UI right inside Colab |

---

## 🔄 Agent Workflow

```
┌─────────────────────────────────────────────────────────┐
│                    USER INPUT (Gradio UI)                │
│              Topic | Audience | Tone | Length           │
└──────────────────────────┬──────────────────────────────┘
                           │
                           ▼
┌──────────────────────────────────────────────────────────┐
│  AGENT 1 — 🔍 Senior Research Analyst                    │
│  • Searches the web using SerperDev                      │
│  • Gathers facts, stats, expert opinions                 │
│  • Outputs a structured research summary                 │
└──────────────────────────┬───────────────────────────────┘
                           │  research summary
                           ▼
┌──────────────────────────────────────────────────────────┐
│  AGENT 2 — ✍️  Professional Content Writer               │
│  • Reads research summary                               │
│  • Writes a structured article with intro/body/conclusion│
│  • Adapts tone and audience from user settings           │
└──────────────────────────┬───────────────────────────────┘
                           │  draft article
                           ▼
┌──────────────────────────────────────────────────────────┐
│  AGENT 3 — ✏️  Senior Editor                             │
│  • Reviews grammar, coherence, and tone                  │
│  • Polishes and finalizes the article                    │
│  • Returns publication-ready output                      │
└──────────────────────────┬───────────────────────────────┘
                           │
                           ▼
              📄 FINAL POLISHED ARTICLE
```

---

## ✅ Prerequisites

Before running this notebook you will need:

1. **OpenAI API Key** → [platform.openai.com](https://platform.openai.com)
2. **Serper API Key** (free tier available) → [serper.dev](https://serper.dev)

> 💡 Both keys are set in **Cell 3**. If you prefer not to use Serper, see the note in that cell for a fallback option.

---
## 📦 Cell 1 — Install Required Libraries

Run this cell **once**. After installation, Colab may prompt you to restart the runtime — go ahead and do so, then continue from **Cell 2**.

| Package | Why we need it |
|---------|----------------|
| `crewai` | Core multi-agent framework |
| `crewai-tools` | Adds SerperDev web search tool |
| `langchain-openai` | OpenAI LLM integration for agents |
| `gradio` | Interactive UI inside Colab |

In [1]:
# ─── Install all required packages ───────────────────────────────────────────
!pip install crewai crewai-tools langchain-openai gradio openai -q

print("✅ All packages installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.9/779.9 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.

---
## 📥 Cell 2 — Import Libraries

We import everything we need up front. Each import is annotated so you know exactly why it's here.

In [2]:
# ─── Standard library ────────────────────────────────────────────────────────
import os
import textwrap

# ─── CrewAI core ─────────────────────────────────────────────────────────────
from crewai import Agent, Task, Crew, Process, LLM   # LLM is CrewAI's native wrapper
from crewai_tools import SerperDevTool                # Web search for Researcher agent

# ─── UI ──────────────────────────────────────────────────────────────────────
import gradio as gr

# ─── NOTE ────────────────────────────────────────────────────────────────────
# CrewAI >= 0.70 uses its own LLM class instead of langchain_openai.ChatOpenAI
# Passing a ChatOpenAI object directly will raise a ValidationError

print("✅ All imports successful!")

✅ All imports successful!


---
## 🔑 Cell 3 — API Key Configuration

Keys are read from **Colab Secrets** (🔑 icon in the left sidebar): add `OPENAI_API_KEY` and `SERPER_API_KEY`. Never hardcode keys in notebooks you share.

In [3]:
# ─── API keys from Colab Secrets (🔑 icon in the left sidebar) ──────────────
# Never hardcode keys in notebooks you share or push to GitHub.
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')   # free key at serper.dev

assert os.environ.get("OPENAI_API_KEY"), "❌ Add OPENAI_API_KEY to Colab Secrets"
print("✅ API keys configured.")

✅ API keys configured.


---
## 🧠 Cell 4 — Initialize the LLM

All three agents share the same GPT-4o model instance. You can tweak:

| Parameter | Effect |
|-----------|--------|
| `model` | Swap to `gpt-4-turbo` or `gpt-3.5-turbo` to reduce cost |
| `temperature` | Lower (0.1) = more factual, Higher (0.8) = more creative |
| `max_tokens` | Cap the response length per agent call |

In [4]:
# ─── Shared LLM for all agents (CrewAI native LLM class) ─────────────────────
# ✅ CrewAI >= 0.70 requires crewai.LLM — NOT langchain_openai.ChatOpenAI
# Passing a ChatOpenAI object raises: ValidationError: llm.BaseLLM
llm = LLM(
    model="gpt-4o-mini",          # Change to 'gpt-4o-mini' or 'gpt-3.5-turbo' to reduce cost
    temperature=0.5,         # 0.0 = deterministic, 1.0 = creative
    max_tokens=500,         # Max tokens per agent response
)

# ─── Web search tool for Researcher ──────────────────────────────────────────
search_tool = SerperDevTool()

print(f"✅ LLM initialized: {llm.model}")
print("✅ SerperDev search tool ready.")

✅ LLM initialized: gpt-4o-mini
✅ SerperDev search tool ready.


---
## 👥 Cell 5 — Define the Three Agents

CrewAI agents are defined by their **role**, **goal**, and **backstory**. These three fields form the system prompt that shapes the agent's behavior. The more specific and realistic you make them, the better the output quality.

### Agent Design Philosophy

```
Role      → What the agent IS  (Senior Research Analyst)
Goal      → What the agent WANTS TO ACHIEVE  (Find accurate information)
Backstory → How the agent THINKS & BEHAVES  (Years of experience, working style)
```

| Agent | Role | Tools | Delegation |
|-------|------|-------|------------|
| 🔍 Researcher | Finds & summarizes information | SerperDev | Off |
| ✍️ Writer | Drafts article from research | None | Off |
| ✏️ Editor | Polishes final article | None | Off |

In [5]:
# ─── AGENT 1: Senior Research Analyst ────────────────────────────────────────
researcher = Agent(
    role="Senior Research Analyst",
    goal=(
        "Find the most accurate, up-to-date, and relevant information on the given topic. "
        "Always prioritize credible sources, cite key statistics, and highlight emerging trends."
    ),
    backstory=(
        "You are a veteran research analyst with 15 years of experience across technology, "
        "business, and science journalism. You have a reputation for digging deep into topics, "
        "cross-referencing multiple sources, and presenting findings in a clear, structured format. "
        "You never make assumptions — everything you report is evidence-backed. "
        "You organize research into clean bullet points with source context."
    ),
    tools=[search_tool],      # Give Researcher access to web search
    llm=llm,
    verbose=True,             # Show agent thinking in output
    allow_delegation=False,   # This agent does not hand off to others
    max_iter=5,               # Max reasoning iterations before giving up
)

# ─── AGENT 2: Professional Content Writer ────────────────────────────────────
writer = Agent(
    role="Professional Content Writer",
    goal=(
        "Transform the research summary into a compelling, well-structured, and audience-appropriate article. "
        "Ensure the article flows naturally, uses clear headings, and maintains the requested tone throughout."
    ),
    backstory=(
        "You are an award-winning content writer with expertise in long-form journalism, technical writing, "
        "and digital publishing. You have written for Forbes, TechCrunch, and Harvard Business Review. "
        "You have a talent for making complex topics accessible without dumbing them down. "
        "You always start with a strong hook, use subheadings to guide the reader, "
        "and end with a memorable conclusion that reinforces the key message."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=4,
)

# ─── AGENT 3: Senior Editor ───────────────────────────────────────────────────
editor = Agent(
    role="Senior Editor",
    goal=(
        "Review and refine the article to ensure it is grammatically flawless, tonally consistent, "
        "logically structured, and ready for publication. Return only the final polished article."
    ),
    backstory=(
        "You are a meticulous senior editor with 20 years of experience at top-tier publications. "
        "You have an unmatched eye for inconsistency, passive voice overuse, redundant phrasing, "
        "and weak transitions. You tighten prose without stripping its personality. "
        "You ensure every paragraph earns its place and every sentence is purposeful. "
        "You never add commentary — you deliver only the clean, final article."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3,
)

print("✅ All 3 agents initialized:")
print(f"   🔍 {researcher.role}")
print(f"   ✍️  {writer.role}")
print(f"   ✏️  {editor.role}")

✅ All 3 agents initialized:
   🔍 Senior Research Analyst
   ✍️  Professional Content Writer
   ✏️  Senior Editor


---
## 📋 Cell 6 — Task Builder Function

Tasks are the units of work each agent executes. In CrewAI, tasks are assigned to specific agents and can have **context** — meaning they can read the output of previous tasks.

We wrap task creation inside a function so that user inputs (topic, audience, tone, length) are baked into the task descriptions dynamically.

```
Task 1 (Researcher) → produces research_output
Task 2 (Writer)     → reads research_output as context
Task 3 (Editor)     → reads writing_output as context
```

In [6]:
def build_tasks(topic: str, audience: str, tone: str, length: str) -> list:
    """
    Dynamically build the 3 tasks with user inputs baked in.
    Returns a list of Task objects in execution order.
    """

    # ── TASK 1: Research ──────────────────────────────────────────────────────
    research_task = Task(
        description=textwrap.dedent(f"""
            Conduct thorough research on the following topic:
            TOPIC: \"{topic}\"

            Your research should be targeted for this audience: {audience}
            Required depth: {length}

            Your deliverables:
            1. A 2-sentence topic overview
            2. 5–8 key facts or statistics with source context
            3. 3–5 recent developments or trends (2023–2025)
            4. 2–3 expert opinions or notable perspectives
            5. A list of 3 key takeaways

            Format everything in clearly labelled bullet points.
        """),
        expected_output=(
            "A comprehensive, well-structured research summary in bullet points "
            "covering overview, key facts, recent developments, expert opinions, and takeaways."
        ),
        agent=researcher,
    )

    # ── TASK 2: Write ─────────────────────────────────────────────────────────
    writing_task = Task(
        description=textwrap.dedent(f"""
            Using the research summary provided, write a complete article on:
            TOPIC: \"{topic}\"

            Specifications:
            - Target Audience: {audience}
            - Tone: {tone}
            - Length: {length}

            Structure:
            1. Headline — compelling and specific
            2. Introduction — hook + why this topic matters now
            3. Section 1 with subheading — background / context
            4. Section 2 with subheading — key developments or insights
            5. Section 3 with subheading — implications or practical applications
            6. Conclusion — summary + forward-looking statement

            Rules:
            - Do NOT use generic filler phrases like "In conclusion" or "It is worth noting"
            - Use active voice whenever possible
            - Every paragraph should add new value — no repetition
        """),
        expected_output=(
            "A complete, structured article with headline, introduction, three body sections "
            "with subheadings, and a conclusion — written in the requested tone for the specified audience."
        ),
        agent=writer,
        context=[research_task],    # Writer reads Researcher's output
    )

    # ── TASK 3: Edit ──────────────────────────────────────────────────────────
    editing_task = Task(
        description=textwrap.dedent(f"""
            You are the final gatekeeper before publication.
            Review and edit the article for the following:

            Checklist:
            ✅ Grammar and spelling are flawless
            ✅ Tone is consistent and matches: {tone}
            ✅ Audience appropriateness: {audience}
            ✅ No redundant sentences or paragraphs
            ✅ Transitions between sections are smooth
            ✅ Headline is punchy and specific
            ✅ Introduction hooks the reader immediately
            ✅ Conclusion ends with impact

            IMPORTANT: Return ONLY the final polished article text.
            Do NOT include your editorial notes, checklist, or any commentary.
        """),
        expected_output=(
            "The final, publication-ready article — clean, polished, and complete. "
            "No editor notes or commentary included."
        ),
        agent=editor,
        context=[writing_task],     # Editor reads Writer's output
    )

    return [research_task, writing_task, editing_task]

print("✅ Task builder function ready.")

✅ Task builder function ready.


---
## 🚀 Cell 7 — Crew Runner Function

This is the main orchestration function. It:
1. Validates the user input
2. Builds the 3 tasks with user parameters
3. Assembles the Crew
4. Kicks off the sequential execution pipeline
5. Returns the final article string

### `Process.sequential` vs `Process.hierarchical`

| Mode | How it works | Best for |
|------|-------------|----------|
| `sequential` | Agents run one after another in order | Linear pipelines like ours |
| `hierarchical` | A manager agent assigns and oversees tasks | Complex dynamic workflows |

In [7]:
def run_crew(topic: str, audience: str, tone: str, length: str) -> str:
    """
    Main orchestration function.
    Builds tasks, assembles the crew, and returns the final article.

    Args:
        topic    : The subject to research and write about
        audience : Target reader profile
        tone     : Writing style (e.g. Informative, Persuasive)
        length   : Article length preference

    Returns:
        str: The final polished article from the Editor agent
    """

    # ── Input validation ──────────────────────────────────────────────────────
    if not topic or not topic.strip():
        return "⚠️ Please enter a topic before generating."

    if len(topic.strip()) < 5:
        return "⚠️ Topic is too short. Please be more descriptive (e.g. 'AI in healthcare 2025')."

    print(f"\n{'='*60}")
    print(f"🚀 Starting CrewAI Pipeline")
    print(f"   Topic    : {topic}")
    print(f"   Audience : {audience}")
    print(f"   Tone     : {tone}")
    print(f"   Length   : {length}")
    print(f"{'='*60}\n")

    try:
        # ── Build dynamic tasks ───────────────────────────────────────────────
        tasks = build_tasks(topic, audience, tone, length)

        # ── Assemble the crew ─────────────────────────────────────────────────
        crew = Crew(
            agents=[researcher, writer, editor],
            tasks=tasks,
            process=Process.sequential,  # Agents run in order: R → W → E
            verbose=True,                # Show full agent logs
            memory=False,                # No cross-run memory (stateless per run)
        )

        # ── Execute ───────────────────────────────────────────────────────────
        print("\n🔍 Agent 1 (Researcher) starting...\n")
        result = crew.kickoff()

        print(f"\n{'='*60}")
        print("✅ CrewAI pipeline complete!")
        print(f"{'='*60}\n")

        return str(result)

    except Exception as e:
        error_msg = f"❌ An error occurred during pipeline execution:\n{str(e)}"
        print(error_msg)
        return error_msg

print("✅ run_crew() function ready.")

✅ run_crew() function ready.


---
## 🧪 Cell 8 — Quick Test (No UI)

Run this cell to test the full pipeline directly in the notebook **before** launching the Gradio UI. This is useful for debugging agent behavior and verifying your API keys work correctly.

Comment this cell out once you're happy and move to the Gradio UI in Cell 9.

In [8]:
# ─── Quick test run ───────────────────────────────────────────────────────────
# Uncomment to test without the UI

test_result = run_crew(
    topic="The rise of AI agents in 2025",
    audience="Tech Enthusiasts",
    tone="Informative",
    length="Medium (600–900 words)"
)
print(test_result)


🚀 Starting CrewAI Pipeline
   Topic    : The rise of AI agents in 2025
   Audience : Tech Enthusiasts
   Tone     : Informative
   Length   : Medium (600–900 words)


🔍 Agent 1 (Researcher) starting...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b79b8d2b-3fd0-4fa7-9f21-ee294f11709e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Conduct thorough research on the following topic:                                                              │
│  TOPIC: "The rise of AI agents in 2025"                                                                         │
│                                                                                                                 │
│  Your research should be targeted for this audience: Tech Enthusiasts                                           │
│  Required depth: Medium (600–900 words)                                                                         │
│                                                                                                                 │
│  Your deliverables:                                                                                             │
│  1. A 2-sentence topic overview                                                                                 │
│  2. 5–8 key facts or statistics with source context                                                             │
│  3. 3–5 recent developments or trends (2023–2025)                                                               │
│  4. 2–3 expert opinions or notable perspectives                                                                 │
│  5. A list of 3 key takeaways                                                                                   │
│                                                                                                                 │
│  Format everything in clearly labelled bullet points.                                                           │
│                                                                                                                 │
│  ID: 217fb5fa-8759-4bdb-8c3b-2e3363b99988                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Conduct thorough research on the following topic:                                                              │
│  TOPIC: "The rise of AI agents in 2025"                                                                         │
│                                                                                                                 │
│  Your research should be targeted for this audience: Tech Enthusiasts                                           │
│  Required depth: Medium (600–900 words)                                                                         │
│                                                                                                                 │
│  Your deliverables:                                                                                             │
│  1. A 2-sentence topic overview                                                                                 │
│  2. 5–8 key facts or statistics with source context                                                             │
│  3. 3–5 recent developments or trends (2023–2025)                                                               │
│  4. 2–3 expert opinions or notable perspectives                                                                 │
│  5. A list of 3 key takeaways                                                                                   │
│                                                                                                                 │
│  Format everything in clearly labelled bullet points.                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'The rise of AI agents in 2025'}                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  You are the final gatekeeper before publication.                                                               │
│  Review and edit the article for the following:                                                                 │
│                                                                                                                 │
│  Checklist:                                                                                                     │
│  ✅ Grammar and spelling are flawless                                                                           │
│  ✅ Tone is consistent and matches: Informative                                                                 │
│  ✅ Audience appropriateness: Tech Enthusiasts                                                                  │
│  ✅ No redundant sentences or paragraphs                                                                        │
│  ✅ Transitions between sections are smooth                                                                     │
│  ✅ Headline is punchy and specific                                                                             │
│  ✅ Introduction hooks the reader immediately                                                                   │
│  ✅ Conclusion ends with impact                                                                                 │
│                                                                                                                 │
│  IMPORTANT: Return ONLY the final polished article text.                                                        │
│  Do NOT include your editorial notes, checklist, or any commentary.                                             │
│                                                                                                                 │
│  ID: c91063d9-db0c-4110-9c7e-f167356b874b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Editor                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You are the final gatekeeper before publication.                                                               │
│  Review and edit the article for the following:                                                                 │
│                                                                                                                 │
│  Checklist:                                                                                                     │
│  ✅ Grammar and spelling are flawless                                                                           │
│  ✅ Tone is consistent and matches: Informative                                                                 │
│  ✅ Audience appropriateness: Tech Enthusiasts                                                                  │
│  ✅ No redundant sentences or paragraphs                                                                        │
│  ✅ Transitions between sections are smooth                                                                     │
│  ✅ Headline is punchy and specific                                                                             │
│  ✅ Introduction hooks the reader immediately                                                                   │
│  ✅ Conclusion ends with impact                                                                                 │
│                                                                                                                 │
│  IMPORTANT: Return ONLY the final polished article text.                                                        │
│  Do NOT include your editorial notes, checklist, or any commentary.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Editor                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # The Rise of AI Agents: A Transformative Shift by 2025                                                        │
│                                                                                                                 │
│  As we stand on the cusp of a new era in technology, the rise of AI agents is set to redefine the landscape of  │
│  business and personal productivity by 2025. These autonomous entities promise to revolutionize                 │
│  decision-making processes, enhance operational efficiencies, and reshape creative endeavors across various     │
│  sectors. As organizations grapple with ever-increasing demands for efficiency and innovation, understanding    │
│  the trajectory of AI agents is not just timely—it's essential.                                                 │
│                                                                                                                 │
│  ## The Context of AI Agents in Today's World                                                                   │
│                                                                                                                 │
│  Artificial intelligence has evolved from a niche technology to a cornerstone of modern business practices. By  │
│  2025, the market for AI agents is projected to experience explosive growth, soaring from USD 7.84 billion in   │
│  2025 to an astounding USD 52.62 billion by 2030, reflecting a compound annual growth rate (CAGR) of 46.3%      │
│  (Source: MarketsandMarkets). This rapid expansion underscores the urgency for businesses to adapt to these     │
│  advancements. As of 2023, only 32% of companies had deployed generative AI agents in at least one function,    │
│  but this figure is expected to leap to 72% by 2025 (Source: McKinsey). Such a shift indicates not only a       │
│  growing acceptance of AI technology but also a recognition of its potential to drive substantial improvements  │
│  in productivity and decision-making.                                                                           │
│                                                                                                                 │
│  The rise of AI agents is not merely a trend; it signifies a fundamental change in how businesses operate.      │
│  With 62% of organizations currently experimenting with AI agents, the appetite for innovation is palpable      │
│  (Source: McKinsey). Companies are beginning to understand that integrating AI into core business processes     │
│  can yield significant benefits, especially in customer service, supply chain management, and marketing         │
│  (Source: Codewave). This movement towards AI is not just about automation; it’s about creating a more          │
│  intelligent and responsive business environment.                                                               │
│                                                                                                                 │
│  ## Key Developments Shaping the Future                                                                         │
│                                                                                                                 │
│  Several noteworthy developments are shaping the capabilities of AI agents as we approach 2025. One of the      │
│  most significant advancements is the emergence of auto

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  You are the final gatekeeper before publication.                                                               │
│  Review and edit the article for the following:                                                                 │
│                                                                                                                 │
│  Checklist:                                                                                                     │
│  ✅ Grammar and spelling are flawless                                                                           │
│  ✅ Tone is consistent and matches: Informative                                                                 │
│  ✅ Audience appropriateness: Tech Enthusiasts                                                                  │
│  ✅ No redundant sentences or paragraphs                                                                        │
│  ✅ Transitions between sections are smooth                                                                     │
│  ✅ Headline is punchy and specific                                                                             │
│  ✅ Introduction hooks the reader immediately                                                                   │
│  ✅ Conclusion ends with impact                                                                                 │
│                                                                                                                 │
│  IMPORTANT: Return ONLY the final polished article text.                                                        │
│  Do NOT include your editorial notes, checklist, or any commentary.                                             │
│                                                                                                                 │
│  Agent: Senior Editor                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ CrewAI pipeline complete!

# The Rise of AI Agents: A Transformative Shift by 2025

As we stand on the cusp of a new era in technology, the rise of AI agents is set to redefine the landscape of business and personal productivity by 2025. These autonomous entities promise to revolutionize decision-making processes, enhance operational efficiencies, and reshape creative endeavors across various sectors. As organizations grapple with ever-increasing demands for efficiency and innovation, understanding the trajectory of AI agents is not just timely—it's essential.

## The Context of AI Agents in Today's World

Artificial intelligence has evolved from a niche technology to a cornerstone of modern business practices. By 2025, the market for AI agents is projected to experience explosive growth, soaring from USD 7.84 billion in 2025 to an astounding USD 52.62 billion by 2030, reflecting a compound annual growth rate (CAGR) of 46.3% (Source: MarketsandMarkets). This rapid expansion undersco

---
## 🎨 Cell 9 — Launch Gradio UI

This cell builds and launches the full interactive interface. Key UI features:

| Feature | Details |
|---------|---------|
| **Topic input** | Free-text field for any subject |
| **Audience selector** | 5 audience types to match tone |
| **Tone selector** | 5 writing styles |
| **Length selector** | Short / Medium / Long |
| **Progress indicator** | Shows which agent is running |
| **Copy button** | One-click copy of the final article |
| **Public URL** | `share=True` gives a temporary public link |

> ⏱️ **Expected runtime:** 45–120 seconds depending on topic complexity and OpenAI API speed.

In [ ]:
# ─── Gradio UI ────────────────────────────────────────────────────────────────

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate"),
    title="🤖 Multi-Agent Article Generator",
    css="""
        .title-text { text-align: center; }
        .agent-badge { font-size: 0.85rem; color: #666; }
        footer { display: none !important; }
    """
) as demo:

    # ── Header ────────────────────────────────────────────────────────────────
    gr.Markdown(
        """
        <div class="title-text">
        <h1>🤖 Multi-Agent Article Generator</h1>
        <p style="color:#555;">Powered by <strong>CrewAI</strong> + <strong>GPT-4o</strong> + <strong>SerperDev</strong></p>
        </div>

        ---

        ### 🔄 How it works
        `🔍 Researcher` searches the web and summarizes findings →
        `✍️ Writer` drafts a structured article →
        `✏️ Editor` polishes and finalizes for publication

        > ⏱️ Generation takes **45–120 seconds**. Agent logs appear in the Colab console below.
        """
    )

    # ── Input + Output layout ─────────────────────────────────────────────────
    with gr.Row(equal_height=False):

        # LEFT: Inputs
        with gr.Column(scale=1, min_width=280):
            gr.Markdown("### ⚙️ Settings")

            topic_input = gr.Textbox(
                label="📌 Topic *",
                placeholder="e.g. The impact of AI agents on software development in 2025",
                lines=3,
                info="Be specific — better topic = better article."
            )

            audience_input = gr.Dropdown(
                label="👥 Target Audience",
                choices=[
                    "General Public",
                    "Business Professionals",
                    "Students & Learners",
                    "Researchers & Academics",
                    "Tech Enthusiasts"
                ],
                value="General Public",
                info="Shapes vocabulary and depth of explanation."
            )

            tone_input = gr.Dropdown(
                label="🎨 Writing Tone",
                choices=[
                    "Informative",
                    "Persuasive",
                    "Conversational",
                    "Academic",
                    "Inspirational"
                ],
                value="Informative",
                info="Controls the style and voice of the article."
            )

            length_input = gr.Radio(
                label="📏 Article Length",
                choices=[
                    "Short (300–500 words)",
                    "Medium (600–900 words)",
                    "Long (1000–1400 words)"
                ],
                value="Medium (600–900 words)",
            )

            gr.Markdown("---")

            submit_btn = gr.Button(
                "🚀 Generate Article",
                variant="primary",
                size="lg"
            )

            clear_btn = gr.Button(
                "🗑️ Clear",
                variant="secondary",
                size="sm"
            )

            gr.Markdown(
                """
                <div class="agent-badge">
                <b>Agents:</b><br>
                🔍 Researcher → web search + summarize<br>
                ✍️ Writer → draft structured article<br>
                ✏️ Editor → polish + finalize
                </div>
                """
            )

        # RIGHT: Output
        with gr.Column(scale=2):
            gr.Markdown("### 📄 Generated Article")

            output_box = gr.Textbox(
                label="",
                lines=35,
                placeholder=(
                    "Your article will appear here once generation is complete.\n\n"
                    "⏱️ This takes 45–120 seconds.\n"
                    "📋 Watch the Colab console below for live agent logs."
                ),
                show_copy_button=True,
                interactive=False,
            )

    # ── Examples ──────────────────────────────────────────────────────────────
    gr.Markdown("### 💡 Example Topics")
    gr.Examples(
        examples=[
            ["The future of AI agents in enterprise software", "Business Professionals", "Informative", "Medium (600–900 words)"],
            ["How large language models are changing education", "Students & Learners", "Conversational", "Short (300–500 words)"],
            ["Quantum computing: where are we in 2025?", "Tech Enthusiasts", "Informative", "Long (1000–1400 words)"],
            ["The ethics of autonomous AI decision-making", "Researchers & Academics", "Academic", "Long (1000–1400 words)"],
            ["Why every startup needs an AI strategy in 2025", "Business Professionals", "Persuasive", "Medium (600–900 words)"],
        ],
        inputs=[topic_input, audience_input, tone_input, length_input],
        label="Click any example to auto-fill the inputs"
    )

    # ── Footer info ───────────────────────────────────────────────────────────
    gr.Markdown(
        """
        ---
        <div style="text-align:center; color:#888; font-size:0.85rem;">
        Built with <strong>CrewAI</strong> · <strong>LangChain</strong> · <strong>GPT-4o</strong> · <strong>Gradio</strong>
        </div>
        """
    )

    # ── Button actions ────────────────────────────────────────────────────────
    submit_btn.click(
        fn=run_crew,
        inputs=[topic_input, audience_input, tone_input, length_input],
        outputs=output_box,
        api_name="generate",
    )

    clear_btn.click(
        fn=lambda: ("", ""),
        inputs=[],
        outputs=[topic_input, output_box]
    )

# ── Launch ────────────────────────────────────────────────────────────────────
demo.launch(
    share=True,        # Generates a public shareable URL (valid for 72 hrs)
    debug=True,        # Shows errors in the UI instead of just the console
    show_error=True,   # Surface any Python errors directly in Gradio output
)

/tmp/ipykernel_398/588107062.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_398/588107062.py:3: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a6e0b23addcdb6e119.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🚀 Starting CrewAI Pipeline
   Topic    : US Iran negotiations
   Audience : General Public
   Tone     : Informative
   Length   : Short (300–500 words)


🔍 Agent 1 (Researcher) starting...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3ac37afd-2976-41c1-929d-dfd7affa0f43                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Conduct thorough research on the following topic:                                                              │
│  TOPIC: "US Iran negotiations"                                                                                  │
│                                                                                                                 │
│  Your research should be targeted for this audience: General Public                                             │
│  Required depth: Short (300–500 words)                                                                          │
│                                                                                                                 │
│  Your deliverables:                                                                                             │
│  1. A 2-sentence topic overview                                                                                 │
│  2. 5–8 key facts or statistics with source context                                                             │
│  3. 3–5 recent developments or trends (2023–2025)                                                               │
│  4. 2–3 expert opinions or notable perspectives                                                                 │
│  5. A list of 3 key takeaways                                                                                   │
│                                                                                                                 │
│  Format everything in clearly labelled bullet points.                                                           │
│                                                                                                                 │
│  ID: 87d1efa1-57ab-4fe7-941d-43f4f166ceec                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Editor                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Navigating the Complex Waters of U.S.-Iran Negotiations in 2025                                              │
│                                                                                                                 │
│  As the world watches, the negotiations between the United States and Iran have taken center stage in           │
│  international diplomacy, particularly regarding Iran's nuclear ambitions and regional stability. With rising   │
│  tensions and a shifting political landscape, the outcome of these talks could have far-reaching implications   │
│  not just for the two nations involved but for global security as a whole.                                      │
│                                                                                                                 │
│  ## Background: A Longstanding Diplomatic Challenge                                                             │
│                                                                                                                 │
│  The history of U.S.-Iran relations is fraught with tension, particularly since the 1979 Iranian Revolution.    │
│  In recent years, the spotlight has intensified on Iran's nuclear program, with the U.S. and its allies         │
│  expressing concerns that it could lead to the development of nuclear weapons—an assertion Iran vehemently      │
│  denies (Source: Arms Control Association). On April 12, 2025, direct negotiations resumed in Muscat, Oman,     │
│  marking a significant diplomatic initiative aimed at resolving these issues. This renewed dialogue comes at a  │
│  time when public sentiment in the U.S. leans heavily toward diplomacy; a Gallup poll indicates that 70% of     │
│  Americans favor negotiations with Iran (Source: Gallup).                                                       │
│                                                                                                                 │
│  ## Key Developments: Shifting Dynamics                                                                         │
│                                                                                                                 │
│  Recent developments have significantly influenced the negotiations. The election of a new Iranian president    │
│  in 2025, who advocates for diplomatic engagement, has created a more favorable atmosphere for talks (Source:   │
│  PBS). However, the path to a comprehensive agreement remains rocky. Despite multiple rounds of negotiations,   │
│  both sides have struggled to meet each other's demands, leading to a stalemate (Source: The Bulletin).         │
│  Complicating matters further, the U.S. military has announced plans to block Iranian shipping, signaling a     │
│  potential escalation in tensions (Source: Reuters). Meanwhile, increased military collaboration between the    │
│  U.S. and Israel has raised alarms in Tehran, further complicating the diplomatic landscape (Source: House of   │
│  Commons Library).                                                                                              │
│                                                                                                                 │
│  Public protests within Iran, spurred by economic hardships attributed to international sanctions, have also    │
│  influenced the government's negotiating stance. Irania

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  You are the final gatekeeper before publication.                                                               │
│  Review and edit the article for the following:                                                                 │
│                                                                                                                 │
│  Checklist:                                                                                                     │
│  ✅ Grammar and spelling are flawless                                                                           │
│  ✅ Tone is consistent and matches: Informative                                                                 │
│  ✅ Audience appropriateness: General Public                                                                    │
│  ✅ No redundant sentences or paragraphs                                                                        │
│  ✅ Transitions between sections are smooth                                                                     │
│  ✅ Headline is punchy and specific                                                                             │
│  ✅ Introduction hooks the reader immediately                                                                   │
│  ✅ Conclusion ends with impact                                                                                 │
│                                                                                                                 │
│  IMPORTANT: Return ONLY the final polished article text.                                                        │
│  Do NOT include your editorial notes, checklist, or any commentary.                                             │
│                                                                                                                 │
│  Agent: Senior Editor                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ CrewAI pipeline complete!



╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3ac37afd-2976-41c1-929d-dfd7affa0f43                                                                       │
│  Final Output: # Navigating the Complex Waters of U.S.-Iran Negotiations in 2025                                │
│                                                                                                                 │
│  As the world watches, the negotiations between the United States and Iran have taken center stage in           │
│  international diplomacy, particularly regarding Iran's nuclear ambitions and regional stability. With rising   │
│  tensions and a shifting political landscape, the outcome of these talks could have far-reaching implications   │
│  not just for the two nations involved but for global security as a whole.                                      │
│                                                                                                                 │
│  ## Background: A Longstanding Diplomatic Challenge                                                             │
│                                                                                                                 │
│  The history of U.S.-Iran relations is fraught with tension, particularly since the 1979 Iranian Revolution.    │
│  In recent years, the spotlight has intensified on Iran's nuclear program, with the U.S. and its allies         │
│  expressing concerns that it could lead to the development of nuclear weapons—an assertion Iran vehemently      │
│  denies (Source: Arms Control Association). On April 12, 2025, direct negotiations resumed in Muscat, Oman,     │
│  marking a significant diplomatic initiative aimed at resolving these issues. This renewed dialogue comes at a  │
│  time when public sentiment in the U.S. leans heavily toward diplomacy; a Gallup poll indicates that 70% of     │
│  Americans favor negotiations with Iran (Source: Gallup).                                                       │
│                                                                                                                 │
│  ## Key Developments: Shifting Dynamics                                                                         │
│                                                                                                                 │
│  Recent developments have significantly influenced the negotiations. The election of a new Iranian president    │
│  in 2025, who advocates for diplomatic engagement, has created a more favorable atmosphere for talks (Source:   │
│  PBS). However, the path to a comprehensive agreement remains rocky. Despite multiple rounds of negotiations,   │
│  both sides have struggled to meet each other's demands, leading to a stalemate (Source: The Bulletin).         │
│  Complicating matters further, the U.S. military has announced plans to block Iranian shipping, signaling a     │
│  potential escalation in tensions (Source: Reuters). Meanwhile, increased military collaboration between the    │
│  U.S. and Israel has raised alarms in Tehran, further complicating the diplomatic landscape (Source: House of   │
│  Commons Library).                                                                                              │
│                                                                                                                 │
│  Public protests within Iran, spurred by economic hardships attributed to international sanctions, have also    │
│  influenced the government's negotiating stance. Irani

---
## 💡 Cell 10 — Tips, Troubleshooting & Extensions

### 🐛 Common Issues

| Issue | Fix |
|-------|-----|
| `AuthenticationError` | Double-check your `OPENAI_API_KEY` in Cell 3 |
| `RateLimitError` | You've hit OpenAI's rate limit — wait 60s or upgrade your plan |
| Serper errors | Check `SERPER_API_KEY` or remove `tools=[search_tool]` to skip web search |
| Slow generation | Normal — 3 agents × multiple LLM calls = 60–120s total |
| Empty output | Increase `max_iter` on the failing agent or simplify the topic |

### 🔧 Easy Customizations

**Change the LLM model** (Cell 4):
```python
llm = LLM(model="gpt-4o-mini",  temperature=0.4)  # Cheaper, faster
llm = LLM(model="gpt-4-turbo",  temperature=0.3)  # Most powerful
llm = LLM(model="gpt-3.5-turbo",temperature=0.4)  # Budget option
```

**Add a 4th agent — SEO Optimizer:**
```python
seo_agent = Agent(
    role="SEO Specialist",
    goal="Optimize the article for search engine ranking without reducing quality.",
    backstory="You are an SEO expert who improves keyword density, meta descriptions, and headings.",
    llm=llm,
    verbose=True,
)
```

**Switch to Hierarchical process** for dynamic task assignment:
```python
crew = Crew(
    agents=[researcher, writer, editor],
    tasks=tasks,
    process=Process.hierarchical,
    manager_llm=llm,   # Required for hierarchical mode
)
```

### 🚀 Next Steps
- Add a **PDF export** button to the Gradio UI
- Integrate a **social media post generator** agent as a 4th step
- Connect **LangGraph** to add conditional routing based on article quality score
- Deploy to **Hugging Face Spaces** for a permanent public URL